# Paper figures — exact Stage-1 task and perturbation gallery

Renders the policy-facing agent-view reset frame for the three exact Stage-1 tasks under ID and all seven frozen perturbations. The exact query supplied to the policy is printed under each image. No VLA checkpoint is loaded.


In [ ]:
import os, subprocess
from pathlib import Path

GPU = "4"  # physical GPU used only for MuJoCo EGL; no policy model is loaded
R = Path.home() / "async-vla-latency-bench"
P = Path.home() / "LIBERO-plus"
N = Path.home() / "stage1-native"
ID = Path.home() / "venv-stage1-id/bin/python"
OOD = Path.home() / "venv-stage1-ood/bin/python"
VAR = R / "paper_figures/source/stage1_resolved_variants.csv"
OUT = Path.home() / "stage1_task_gallery"
CFG = R / "async_vla_benchmark/configs/stage1.yaml"
OUT.mkdir(exist_ok=True)
for path in (R, P, P / "libero/libero/assets", N / "lib", ID, OOD, VAR, CFG):
    if not path.exists():
        raise SystemExit(f"STOP: missing prerequisite {path}")
state = subprocess.run(["nvidia-smi", "-i", GPU, "--query-gpu=name,memory.used,utilization.gpu", "--format=csv,noheader,nounits"], capture_output=True, text=True, check=True).stdout.strip()
print("EGL GPU:", state)
print("PASS: gallery prerequisites available; no model checkpoint will be loaded")


In [ ]:
base = os.environ.copy()
base.update({"CUDA_VISIBLE_DEVICES": GPU, "MUJOCO_EGL_DEVICE_ID": GPU, "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl", "MPLBACKEND": "Agg"})
command = [str(ID), "-u", "-m", "async_vla_benchmark.scripts.render_stage1_task_gallery", "--scene", "id", "--config", str(CFG), "--variants", str(VAR), "--output-dir", str(OUT), "--seed", "0"]
subprocess.run(command, cwd=R, env=base, check=True)
print("PASS: rendered the three exact ID frames")


In [ ]:
ood = base.copy()
ood.update({"PYTHONPATH": str(P), "MAGICK_HOME": str(N), "PATH": str(N / "bin") + os.pathsep + ood.get("PATH", ""), "LD_LIBRARY_PATH": str(N / "lib") + os.pathsep + ood.get("LD_LIBRARY_PATH", "")})
command = [str(OOD), "-u", "-m", "async_vla_benchmark.scripts.render_stage1_task_gallery", "--scene", "ood", "--config", str(CFG), "--variants", str(VAR), "--output-dir", str(OUT), "--seed", "0"]
subprocess.run(command, cwd=R, env=ood, check=True)
print("PASS: rendered all 21 exact OOD frames")


In [ ]:
import hashlib
command = [str(ID), "-u", "-m", "async_vla_benchmark.scripts.render_stage1_task_gallery", "--scene", "compose", "--config", str(CFG), "--variants", str(VAR), "--output-dir", str(OUT)]
subprocess.run(command, cwd=R, env=base, check=True)
frames = sorted((OUT / "frames").glob("*.png"))
assert len(frames) == 24, f"STOP: expected 24 frames, found {len(frames)}"
archive = Path.home() / "stage1_task_gallery.tar.gz"
subprocess.run(["tar", "-czf", str(archive), "-C", str(Path.home()), OUT.name], check=True)
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
print(OUT / "stage1_exact_task_perturbation_gallery.png")
print(OUT / "stage1_exact_task_perturbation_gallery.pdf")
print(archive, archive.stat().st_size, "bytes")
print("sha256", digest)
print("DOWNLOAD THE GALLERY ARCHIVE NOW")
